# Hackathon AI & ML — Advanced v5
**Base v3 (F1=0.7108) + ExtraTrees comme 5ème modèle diverse**

Changements vs v3 :
- ExtraTrees ajouté (splits aléatoires → diversité orthogonale aux boosting)
- Nelder-Mead étendu à 5 modèles
- Stacking LogisticRegression + calibration isotonique (inchangé, fiable)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import minimize

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

DATA_DIR          = Path('data')
GROUP_ID          = 'G1'
SUBMISSION_ID     = '4'
RANDOM_STATE      = 42
N_FOLDS           = 5
MISSING_THRESHOLD = 0.70

print('Librairies chargées OK')

## 1. Chargement des données

In [ ]:
data     = pd.read_csv(DATA_DIR / 'data.csv')
labels   = pd.read_csv(DATA_DIR / 'ground_truth_train.csv')
test_idx = pd.read_csv(DATA_DIR / 'test_indexes.csv', header=None, names=['SEQN'])
metadata = pd.read_csv(DATA_DIR / 'features_metadata.csv')

test_seqn  = set(test_idx['SEQN'].values)
train_seqn = set(labels['SEQN'].values) - test_seqn

train_data = data[data['SEQN'].isin(train_seqn)].copy()
test_data  = data[data['SEQN'].isin(test_seqn)].copy()
train_df   = train_data.merge(labels[labels['SEQN'].isin(train_seqn)], on='SEQN')

print(f'Train : {train_df.shape}  |  Test : {test_data.shape}')
print(f'Taux de mortalité : {train_df["MORTSTAT_2019"].mean():.2%}')

## 2. Feature engineering

In [ ]:
meta_cols  = metadata[['SAS', 'Component']].rename(columns={'SAS': 'col'})
labo_cols  = meta_cols[meta_cols['Component'] == 'Laboratory']['col'].tolist()
exam_cols  = meta_cols[meta_cols['Component'] == 'Examination']['col'].tolist()
quest_cols = meta_cols[meta_cols['Component'] == 'Questionnaire']['col'].tolist()

def get_func_cols(metadata, keyword):
    return metadata[metadata['function'].apply(lambda f: keyword in str(f))]['SAS'].tolist()

func_groups = {
    'vitality':  get_func_cols(metadata, 'Vitalité'),
    'mobility':  get_func_cols(metadata, 'Mobilité'),
    'cognition': get_func_cols(metadata, 'Cognition'),
    'sensory':   get_func_cols(metadata, 'Sensoriel'),
    'psycho':    get_func_cols(metadata, 'Psychologique'),
}

def add_features(df):
    all_cols = [c for c in df.columns if c not in ['SEQN', 'MORTSTAT_2019']]
    df = df.copy()
    df['feat_nb_missing_total']  = df[all_cols].isnull().sum(axis=1)
    df['feat_pct_missing_total'] = df['feat_nb_missing_total'] / len(all_cols)
    for name, cols in [('labo', labo_cols), ('exam', exam_cols), ('quest', quest_cols)]:
        present = [c for c in cols if c in df.columns]
        if present:
            df[f'feat_nb_missing_{name}'] = df[present].isnull().sum(axis=1)
    for name, cols in func_groups.items():
        present = [c for c in cols if c in df.columns]
        if present:
            df[f'feat_{name}_mean']    = df[present].mean(axis=1)
            df[f'feat_{name}_missing'] = df[present].isnull().mean(axis=1)
    return df

train_df  = add_features(train_df)
test_data = add_features(test_data)
print('Feature engineering OK')

## 3. Nettoyage

In [ ]:
missing_ratio = train_df.isnull().mean()
cols_to_drop  = [c for c in missing_ratio[missing_ratio > MISSING_THRESHOLD].index
                 if c not in ['SEQN', 'MORTSTAT_2019']]

train_clean = train_df.drop(columns=cols_to_drop)
test_clean  = test_data.drop(columns=[c for c in cols_to_drop if c in test_data.columns])

feature_cols    = [c for c in train_clean.columns if c not in ['SEQN', 'MORTSTAT_2019']]
X               = train_clean[feature_cols]
y               = train_clean['MORTSTAT_2019']
X_test          = test_clean[feature_cols]
test_seqn_order = test_clean['SEQN']
neg_pos_ratio   = (y == 0).sum() / (y == 1).sum()

train_medians = X.median()
X_imp      = X.fillna(train_medians)
X_test_imp = X_test.fillna(train_medians)

print(f'Features : {X.shape[1]}  |  Train : {X.shape[0]}  |  Déséquilibre : {neg_pos_ratio:.1f}x')

## 4. Paramètres

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

lgb_params = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'verbose': -1, 'n_jobs': -1, 'random_state': RANDOM_STATE,
    'scale_pos_weight': neg_pos_ratio, 'n_estimators': 2000,
    'learning_rate': 0.023688639503640783, 'num_leaves': 244,
    'min_child_samples': 76, 'subsample': 0.7993292420985183,
    'colsample_bytree': 0.5780093202212182,
    'reg_alpha': 0.004207053950287938, 'reg_lambda': 0.0017073967431528124,
}

xgb_params = {
    'n_estimators': 2000, 'learning_rate': 0.05, 'max_depth': 6,
    'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.6,
    'reg_alpha': 0.01, 'reg_lambda': 1.0,
    'scale_pos_weight': neg_pos_ratio,
    'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbosity': 0,
    'eval_metric': 'logloss', 'early_stopping_rounds': 50,
}

cat_params = {
    'iterations': 1000, 'learning_rate': 0.05, 'depth': 6,
    'loss_function': 'Logloss', 'eval_metric': 'F1',
    'class_weights': [1, neg_pos_ratio],
    'random_seed': RANDOM_STATE, 'verbose': 0, 'early_stopping_rounds': 50,
}

rf_params = {
    'n_estimators': 500, 'max_depth': 20, 'min_samples_leaf': 10,
    'max_features': 'sqrt', 'class_weight': 'balanced',
    'random_state': RANDOM_STATE, 'n_jobs': -1,
}

et_params = {
    'n_estimators': 500, 'max_depth': 20, 'min_samples_leaf': 10,
    'max_features': 'sqrt', 'class_weight': 'balanced',
    'random_state': RANDOM_STATE, 'n_jobs': -1,
}

print('Paramètres OK')

## 5. Entraînement CV — OOF

In [ ]:
oof_lgb  = np.zeros(len(X)); test_lgb = np.zeros(len(X_test))
oof_xgb  = np.zeros(len(X)); test_xgb = np.zeros(len(X_test))
oof_cat  = np.zeros(len(X)); test_cat = np.zeros(len(X_test))
oof_rf   = np.zeros(len(X)); test_rf  = np.zeros(len(X_test))
oof_et   = np.zeros(len(X)); test_et  = np.zeros(len(X_test))

smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
f1_lgb, f1_xgb, f1_cat, f1_rf, f1_et = [], [], [], [], []

print('Début entraînement CV...')

In [ ]:
# --- LightGBM ---
print('=== LightGBM ===')
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[val_idx]  = m.predict_proba(X_val)[:, 1]
    test_lgb         += m.predict_proba(X_test)[:, 1] / N_FOLDS
    f1 = f1_score(y_val, (oof_lgb[val_idx] > 0.5).astype(int))
    f1_lgb.append(f1); print(f'  Fold {fold+1}: {f1:.4f}')
print(f'  LGB CV F1: {np.mean(f1_lgb):.4f} ± {np.std(f1_lgb):.4f}')

In [ ]:
# --- XGBoost ---
print('=== XGBoost ===')
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y)):
    X_tr, X_val = X_imp.iloc[tr_idx], X_imp.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    m = xgb.XGBClassifier(**xgb_params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx]  = m.predict_proba(X_val)[:, 1]
    test_xgb         += m.predict_proba(X_test_imp)[:, 1] / N_FOLDS
    f1 = f1_score(y_val, (oof_xgb[val_idx] > 0.5).astype(int))
    f1_xgb.append(f1); print(f'  Fold {fold+1}: {f1:.4f}')
print(f'  XGB CV F1: {np.mean(f1_xgb):.4f} ± {np.std(f1_xgb):.4f}')

In [ ]:
# --- CatBoost ---
print('=== CatBoost ===')
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    m = CatBoostClassifier(**cat_params)
    m.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    oof_cat[val_idx]  = m.predict_proba(X_val)[:, 1]
    test_cat         += m.predict_proba(X_test)[:, 1] / N_FOLDS
    f1 = f1_score(y_val, (oof_cat[val_idx] > 0.5).astype(int))
    f1_cat.append(f1); print(f'  Fold {fold+1}: {f1:.4f}')
print(f'  CAT CV F1: {np.mean(f1_cat):.4f} ± {np.std(f1_cat):.4f}')

In [ ]:
# --- Random Forest + ExtraTrees + SMOTE ---
print('=== Random Forest + ExtraTrees (SMOTE par fold) ===')
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y)):
    X_tr, X_val = X_imp.iloc[tr_idx], X_imp.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    X_tr_s, y_tr_s = smote.fit_resample(X_tr, y_tr)

    m_rf = RandomForestClassifier(**rf_params)
    m_rf.fit(X_tr_s, y_tr_s)
    oof_rf[val_idx]  = m_rf.predict_proba(X_val)[:, 1]
    test_rf         += m_rf.predict_proba(X_test_imp)[:, 1] / N_FOLDS

    m_et = ExtraTreesClassifier(**et_params)
    m_et.fit(X_tr_s, y_tr_s)
    oof_et[val_idx]  = m_et.predict_proba(X_val)[:, 1]
    test_et         += m_et.predict_proba(X_test_imp)[:, 1] / N_FOLDS

    f_rf = f1_score(y_val, (oof_rf[val_idx] > 0.5).astype(int))
    f_et = f1_score(y_val, (oof_et[val_idx] > 0.5).astype(int))
    f1_rf.append(f_rf); f1_et.append(f_et)
    print(f'  Fold {fold+1}: RF={f_rf:.4f}  ET={f_et:.4f}')
print(f'  RF  CV F1: {np.mean(f1_rf):.4f} ± {np.std(f1_rf):.4f}')
print(f'  ET  CV F1: {np.mean(f1_et):.4f} ± {np.std(f1_et):.4f}')

## 6. Ensemble — poids Nelder-Mead + stacking + calibration

In [ ]:
# --- A. Averaging poids optimisés (5 modèles) ---
def neg_f1_weighted(weights, threshold=0.5):
    w = np.abs(weights) / np.abs(weights).sum()
    oof = w[0]*oof_lgb + w[1]*oof_xgb + w[2]*oof_cat + w[3]*oof_rf + w[4]*oof_et
    return -f1_score(y, (oof > threshold).astype(int))

res  = minimize(neg_f1_weighted, [0.2]*5, method='Nelder-Mead', options={'maxiter': 2000})
opt_w = np.abs(res.x) / np.abs(res.x).sum()

oof_avg  = opt_w[0]*oof_lgb  + opt_w[1]*oof_xgb  + opt_w[2]*oof_cat  + opt_w[3]*oof_rf  + opt_w[4]*oof_et
test_avg = opt_w[0]*test_lgb + opt_w[1]*test_xgb + opt_w[2]*test_cat + opt_w[3]*test_rf + opt_w[4]*test_et

thresholds   = np.arange(0.20, 0.70, 0.01)
f1_avg_thr   = [f1_score(y, (oof_avg > t).astype(int)) for t in thresholds]
best_thr_avg = thresholds[np.argmax(f1_avg_thr)]
best_f1_avg  = max(f1_avg_thr)

print(f'Poids : LGB={opt_w[0]:.3f} XGB={opt_w[1]:.3f} CAT={opt_w[2]:.3f} RF={opt_w[3]:.3f} ET={opt_w[4]:.3f}')
print(f'Averaging OOF F1 (thr={best_thr_avg:.2f}) : {best_f1_avg:.4f}')

In [ ]:
# --- B. Stacking LogisticRegression + calibration isotonique ---
meta_X_train = np.column_stack([oof_lgb, oof_xgb, oof_cat, oof_rf, oof_et])
meta_X_test  = np.column_stack([test_lgb, test_xgb, test_cat, test_rf, test_et])

scaler = StandardScaler()
meta_s = scaler.fit_transform(meta_X_train)
meta_t = scaler.transform(meta_X_test)

meta = LogisticRegression(C=1.0, random_state=RANDOM_STATE, max_iter=1000)
meta.fit(meta_s, y)

iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(meta.predict_proba(meta_s)[:, 1], y)
oof_stack  = iso.transform(meta.predict_proba(meta_s)[:, 1])
test_stack = iso.transform(meta.predict_proba(meta_t)[:, 1])

f1_stack_thr   = [f1_score(y, (oof_stack > t).astype(int)) for t in thresholds]
best_thr_stack = thresholds[np.argmax(f1_stack_thr)]
best_f1_stack  = max(f1_stack_thr)

print(f'Stacking + calibration (thr={best_thr_stack:.2f}) : {best_f1_stack:.4f}')

print('\n=== Résumé ===')
print(f'  Baseline                : 0.6945')
print(f'  v3 (4 modèles)          : 0.7108')
print(f'  LGB                     : {np.mean(f1_lgb):.4f}')
print(f'  XGB                     : {np.mean(f1_xgb):.4f}')
print(f'  CatBoost                : {np.mean(f1_cat):.4f}')
print(f'  RF                      : {np.mean(f1_rf):.4f}')
print(f'  ExtraTrees              : {np.mean(f1_et):.4f}')
print(f'  Averaging (5 modèles)   : {best_f1_avg:.4f}  (thr={best_thr_avg:.2f})')
print(f'  Stacking + calibration  : {best_f1_stack:.4f}  (thr={best_thr_stack:.2f})')

plt.figure(figsize=(9, 4))
plt.plot(thresholds, f1_avg_thr,   label=f'Averaging  (best={best_f1_avg:.4f})')
plt.plot(thresholds, f1_stack_thr, label=f'Stacking+calib (best={best_f1_stack:.4f})')
plt.axvline(best_thr_avg,   color='C0', linestyle='--', alpha=0.6)
plt.axvline(best_thr_stack, color='C1', linestyle='--', alpha=0.6)
plt.xlabel('Threshold'); plt.ylabel('F1 OOF')
plt.title('F1 vs Threshold — v5')
plt.legend(); plt.tight_layout(); plt.show()

## 7. Soumission

In [ ]:
if best_f1_stack >= best_f1_avg:
    best_strategy  = 'Stacking+calibration'
    y_pred_proba   = test_stack
    best_threshold = best_thr_stack
    best_f1        = best_f1_stack
else:
    best_strategy  = 'Averaging'
    y_pred_proba   = test_avg
    best_threshold = best_thr_avg
    best_f1        = best_f1_avg

print(f'Stratégie : {best_strategy} (F1 OOF={best_f1:.4f}, thr={best_threshold:.2f})')

y_pred_test = (y_pred_proba > best_threshold).astype(int)
submission  = pd.DataFrame({'SEQN': test_seqn_order.values, 'prediction': y_pred_test}).sort_values('SEQN')
assert len(submission) == 5000

filename = f'{GROUP_ID}_{SUBMISSION_ID}.csv'
submission.to_csv(filename, index=False, header=False)
print(f'Fichier : {filename}  |  Répartition : {submission["prediction"].value_counts().to_dict()}')